# UFC Fight Predictor — Step 3: Model & honest evaluation
Input: `data/ufc_features.csv` (notebook 2). Output: a logistic regression, an out-of-sample accuracy number, and `data/ufc_test_predictions.csv` for the dashboard later.

**Ground rules for being honest about accuracy**
1. **Split by date, never randomly.** Train on older fights, test on newer ones — exactly the situation of predicting a future card.
2. **Touch the test set once.** Any tuning happens on a validation block *inside* the training period.
3. **Compare against dumb baselines.** 60% sounds good until you learn "pick the fighter with the better record" already gets 57%.
4. **Report uncertainty.** A few hundred fights means the accuracy number has a wide margin of error.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss

pd.set_option("display.width", 200)

MIN_PRIOR = 3          # both fighters need at least this many earlier UFC fights
TEST_START = "2025-01-01"

fights = pd.read_csv("data/ufc_features.csv", parse_dates=["event_date"])
d = fights[fights.min_prev_fights >= MIN_PRIOR].copy().sort_values("event_date").reset_index(drop=True)
print("Modeling rows:", len(d), "|", d.event_date.min().date(), "->", d.event_date.max().date())

## 1. Features
We use the **A-minus-B difference** columns. Positive = fighter A has more of that thing.

Missing values: a difference can only be missing if one fighter's value is missing (mostly reach and takedown stats). We fill with 0, which means "no advantage" and is the neutral choice.

*Deliberately left out: a "reach unknown" flag.* We tested it: it doesn't improve accuracy, and whether a fighter's reach got recorded depends on how long they lasted in the UFC, which is information from after the fight. Features like that are a subtle form of leakage, so we keep them out.

In [ ]:
FEATS = ["diff_n_prev", "diff_prev_win_rate", "diff_last3_win_rate", "diff_streak", "diff_days_since_last",
         "diff_age", "diff_height", "diff_reach", "diff_slpm", "diff_sapm", "diff_sig_acc",
         "diff_td_per15", "diff_td_acc", "diff_td_def", "diff_sub_att_per15", "diff_kd_per15",
         "diff_ctrl_share", "diff_grappling_share", "diff_ko_win_rate", "diff_sub_win_rate"]

print("Missing values per feature:")
print(d[FEATS].isna().sum()[d[FEATS].isna().sum() > 0])

d[FEATS] = d[FEATS].fillna(0)
y = d.a_wins

## 2. Why there is no intercept
Fighter A/B labels are random. If we swapped A and B, every difference flips sign and the outcome flips too. A symmetric model should treat both orderings the same — which means **no intercept** (`fit_intercept=False`). And we scale by spread only (`with_mean=False`) so a difference of 0 stays exactly 0.

In [ ]:
def make_model(C=0.1):
    return make_pipeline(StandardScaler(with_mean=False),
                         LogisticRegression(C=C, fit_intercept=False, max_iter=2000))

def evaluate(y_true, p):
    pred = (p >= 0.5).astype(int)
    return accuracy_score(y_true, pred), log_loss(y_true, p)

def acc_ci(acc, n):
    se = np.sqrt(acc * (1 - acc) / n)
    return acc - 1.96 * se, acc + 1.96 * se

## 3. Pick regularization strength on a *validation* block (inside the training period)
Train on fights before 2022, validate on 2022-2024. The 2025+ test set is not touched.

In [ ]:
train_all = d[d.event_date < TEST_START]
fit_part = train_all[train_all.event_date < "2022-01-01"]
val_part = train_all[train_all.event_date >= "2022-01-01"]

rows = []
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
    m = make_model(C).fit(fit_part[FEATS], fit_part.a_wins)
    acc, ll = evaluate(val_part.a_wins, m.predict_proba(val_part[FEATS])[:, 1])
    rows.append({"C": C, "val_accuracy": round(acc, 3), "val_log_loss": round(ll, 4)})
val_table = pd.DataFrame(rows)
print(val_table.to_string(index=False))
print("Coin-flip log loss for reference: 0.6931")

best_C = val_table.sort_values("val_log_loss").iloc[0]["C"]
print("Chosen C:", best_C)

## 4. Final model: train on everything before 2025, test on 2025 onward

In [ ]:
test = d[d.event_date >= TEST_START].copy()
model = make_model(best_C).fit(train_all[FEATS], train_all.a_wins)

test["p_a_wins"] = model.predict_proba(test[FEATS])[:, 1]
test["pred_a_wins"] = (test.p_a_wins >= 0.5).astype(int)
test["correct"] = (test.pred_a_wins == test.a_wins).astype(int)

acc, ll = evaluate(test.a_wins, test.p_a_wins)
lo, hi = acc_ci(acc, len(test))
print(f"Train fights: {len(train_all)} | Test fights: {len(test)}  ({test.event_date.min().date()} -> {test.event_date.max().date()})")
print(f"Model accuracy: {acc:.1%}   (95% interval roughly {lo:.1%} to {hi:.1%})")
print(f"Model log loss: {ll:.4f}   (coin flip = 0.6931, lower is better)")

## 5. Baselines: is the model actually adding anything?
Two rules a casual fan could apply with no machine learning.

In [ ]:
def rule_accuracy(df, col, higher_wins=True):
    s = df[df[col] != 0]                      # skip exact ties
    pick_a = (s[col] > 0) if higher_wins else (s[col] < 0)
    return (pick_a.astype(int) == s.a_wins).mean(), len(s)

base = pd.DataFrame([
    {"approach": "Coin flip", "accuracy": 0.5, "n": len(test)},
    {"approach": "Pick better career win rate", "accuracy": rule_accuracy(test, "diff_prev_win_rate")[0], "n": rule_accuracy(test, "diff_prev_win_rate")[1]},
    {"approach": "Pick better last-3 win rate", "accuracy": rule_accuracy(test, "diff_last3_win_rate")[0], "n": rule_accuracy(test, "diff_last3_win_rate")[1]},
    {"approach": "Pick the younger fighter", "accuracy": rule_accuracy(test, "diff_age", higher_wins=False)[0], "n": rule_accuracy(test, "diff_age", higher_wins=False)[1]},
    {"approach": "Logistic regression (all features)", "accuracy": acc, "n": len(test)},
])
base["accuracy"] = base.accuracy.round(3)
base

## 6. Is the result stable? Walk-forward by year
Re-train using only data before each year, test on that year. One test window can be lucky or unlucky; many windows tell you more.

In [ ]:
wf = []
for yr in range(2016, test.event_date.dt.year.max() + 1):
    tr = d[d.event_date < f"{yr}-01-01"]
    te = d[(d.event_date >= f"{yr}-01-01") & (d.event_date < f"{yr + 1}-01-01")]
    if len(te) < 30: 
        continue
    m = make_model(best_C).fit(tr[FEATS], tr.a_wins)
    a, l = evaluate(te.a_wins, m.predict_proba(te[FEATS])[:, 1])
    wf.append({"year": yr, "n_test": len(te), "accuracy": a, "log_loss": l})
wf = pd.DataFrame(wf)

pooled = np.average(wf.accuracy, weights=wf.n_test)
print(wf.round(3).to_string(index=False))
print(f"\nPooled walk-forward accuracy: {pooled:.1%} over {wf.n_test.sum()} fights")

plt.figure(figsize=(8, 3.5))
plt.bar(wf.year.astype(str), wf.accuracy)
plt.axhline(0.5, color="red", linestyle="--", label="coin flip")
plt.ylim(0.4, 0.75); plt.ylabel("accuracy"); plt.legend(); plt.title("Walk-forward accuracy by year")
plt.tight_layout(); plt.show()

## 7. When the model is more confident, is it more accurate?
A useful model should be right more often on its high-confidence picks. Otherwise the probabilities are meaningless.

In [ ]:
test["confidence"] = np.maximum(test.p_a_wins, 1 - test.p_a_wins)
test["conf_bucket"] = pd.cut(test.confidence, [0.5, 0.55, 0.6, 0.65, 0.7, 1.0])
test.groupby("conf_bucket", observed=True).agg(fights=("correct", "size"), accuracy=("correct", "mean")).round(3)

## 8. What does the model think matters?
Fit the same kind of model with `statsmodels` to see coefficients and p-values (on standardized features, so sizes are comparable).
**Caution:** several features overlap (career win rate, KO win rate, experience), so individual coefficients are unstable. Use this for a rough picture, not for precise claims.

In [ ]:
scaler = StandardScaler(with_mean=False).fit(train_all[FEATS])
Xs = pd.DataFrame(scaler.transform(train_all[FEATS]), columns=FEATS, index=train_all.index)

logit = sm.Logit(train_all.a_wins, Xs).fit(disp=0)          # no constant added = no intercept
coefs = pd.DataFrame({"coef": logit.params, "p_value": logit.pvalues}).round(4)
coefs["abs_coef"] = coefs.coef.abs()
coefs.sort_values("abs_coef", ascending=False).drop(columns="abs_coef")

## 9. Save predictions (for the dashboard) and coefficients

In [ ]:
out = test[["fight_url", "event_date", "f_1_name", "f_2_name", "weight_class",
            "p_a_wins", "pred_a_wins", "a_wins", "correct"]].rename(
            columns={"f_1_name": "fighter_a", "f_2_name": "fighter_b"})
out.to_csv("data/ufc_test_predictions.csv", index=False)
coefs.drop(columns="abs_coef").to_csv("data/ufc_model_coefficients.csv")
print("Saved", len(out), "predictions and", len(coefs), "coefficients")

## What to take from this
- Compare the model's accuracy to the **baselines** in step 5, not to 100%. The gap between "pick the better record" and the full model is the value that all the feature engineering added.
- Look at the **interval** in step 4 and the **year-by-year spread** in step 6. If years swing by several points, differences of 1-2 points between approaches are noise.
- Step 7 tells you whether the probabilities are trustworthy enough to show on a live dashboard.

**Next (notebook 4):** put the cleaned data and features into Postgres, rewrite the feature logic as SQL window functions, and confirm it reproduces these numbers.